# Fine-tuning de modelo para sumarização de diálogos de atendimento ao cliente pelo Twitter

- Aplica modelos de sumarização (sem fine-tuning) aos diálogos
  
- Avalia a performance dos modelos com a métrica ROUGE

## Configurações iniciais

In [ ]:
import os

import tqdm
from typing import List

from datasets import Dataset
import evaluate
from rich import print
import pandas as pd
from langchain_community.llms import HuggingFacePipeline
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)
import torch

In [3]:
LLM_MODEL_PARAMS = {
    't5_small': {
        'model_name': 'google-t5/t5-small',
        'auto_model_class': AutoModelForSeq2SeqLM,
        'task_pipeline': 'text-generation'
    },
    'falconsai': {
        'model_name': 'Falconsai/text_summarization',
        'auto_model_class': AutoModelForSeq2SeqLM,
        'task_pipeline': 'text-generation'
    }
}

# =============================
MODEL_NAME = 'falconsai'
# MODEL_NAME = 't5_small'


N_MAX_SAMPLES_EVAL = 2327
# =============================


LLM_MODEL_NAME = LLM_MODEL_PARAMS[MODEL_NAME]['model_name']
TASK_PIPLINE = LLM_MODEL_PARAMS[MODEL_NAME]['task_pipeline']
AUTO_MODEL_CLASS = LLM_MODEL_PARAMS[MODEL_NAME]['auto_model_class']

DEVICE = 'cuda'

PRED_COL_NAME = 'summary_pred'

PATH_PREPARED_DATASET_TRAIN = f'../data/interim/summarization_train.csv'
PATH_PREPARED_DATASET_VALID = f'../data/interim/summarization_valid.csv'
PATH_PREPARED_DATASET_TEST = f'../data/interim/summarization_test.csv'

MODEL_PATH = r'/home/msc/Downloads/hf_models'

In [4]:
df_tweets_summ = pd.read_csv(PATH_PREPARED_DATASET_TEST)
print(df_tweets_summ.shape)
df_tweets_summ.head()

(109, 6)

,conversation_id,tweet_ids,created_at_list,tweet_texts,human_summary,elapsed_time
0,bbde6d8ec7c39c4551da1ff6024f997b,"[2263653, 2263654, 2263656, 2263658]",{2263653: Timestamp('2017-11-10 05:21:36+0000'...,@hulu_support My watchlist is not updating wit...,Customer is complaining that the watchlist is ...,6 days 17:39:16
1,1d1a6617ae65baa429c2232ccc908840,"[2172451, 2172451, 2172455]",{2172451: Timestamp('2017-11-09 15:53:28+0000'...,"@AirbnbHelp hi , my Acc was linked to an old n...",Customer is enquiring that whether they can ch...,0 days 03:06:40
2,9555f25de7b6c8dfb8204f56f8bc4dd0,"[1053619, 1053621, 1053622, 1053625]",{1053619: Timestamp('2017-10-23 13:36:53+0000'...,@115858 the new update ios11 sucks. I can’t ev...,Customer is complaining that they are unable t...,0 days 06:32:07
3,54fe18905f0a19ee163a2b452e31e07d,"[2426341, 2426343, 2426346, 2426347]",{2426341: Timestamp('2017-11-14 20:09:00+0000'...,@UPSHelp I’m not DM’ing you because you’re a u...,Customer is complaining about parcel service ...,0 days 00:34:55
4,f6cc57227f74737de08efd03782d015e,"[2880177, 2880180, 2880181, 2880182]",{2880177: Timestamp('2017-11-28 08:14:06+0000'...,"Stuck at Staines waiting for a Reading train, ...",The customer says that he is stuck at Staines ...,0 days 01:15:17


### Checa amostras

In [5]:
idx_test = 1
print(f'Diálogo Twitter:\n{df_tweets_summ.iloc[idx_test].tweet_texts}')
print(f'Sumarização:\n{df_tweets_summ.iloc[idx_test].human_summary}')

Diálogo Twitter:
@AirbnbHelp hi , my Acc was linked to an old number. Now I’m asked to verify my Acc , where a code / call wil be 
sent to my old number. Any way that I can link my Acc to my current number? Pls help @637109 Can you please delete 
this post as it does have personal info in it. We have updated your Case Manager https://t.co/WCQEFGIlXC

Sumarização:
Customer is enquiring that whether they can change the current number to link the account as they have old number 
which was linked. Agent states that they have updated to case manager, they will look into it and also to DM for 
further assistance.

### Define modelo LLM

In [6]:
local_model_path = f'{MODEL_PATH}/{LLM_MODEL_NAME}'

tokenizer = AutoTokenizer.from_pretrained(local_model_path)
model = AUTO_MODEL_CLASS.from_pretrained(
    local_model_path,
    torch_dtype=torch.float32,
    device_map="auto"
)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [7]:
df_eval = df_tweets_summ.head(N_MAX_SAMPLES_EVAL)
df_eval.rename(columns={'tweet_texts': 'inputs'}, inplace=True)
print(df_eval.shape)
df_eval.head(1)

(109, 6)

,conversation_id,tweet_ids,created_at_list,inputs,human_summary,elapsed_time
0,bbde6d8ec7c39c4551da1ff6024f997b,"[2263653, 2263654, 2263656, 2263658]",{2263653: Timestamp('2017-11-10 05:21:36+0000'...,@hulu_support My watchlist is not updating wit...,Customer is complaining that the watchlist is ...,6 days 17:39:16


In [ ]:
def llm_summarization(input_text: str, model=model, tokenizer=tokenizer) -> str:
    prompt = f'summarize: {input_text}'

    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True
    ).to(model.device)

    outputs = model.generate(
        inputs['input_ids'], 
        max_new_tokens=100,
        do_sample=False,
        
        max_length=700, 
        min_length=40,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

In [ ]:
def run_llm_summarization(
    df: pd.DataFrame,
    llm_tokenizer=tokenizer,
    llm_model: HuggingFacePipeline = None,
) -> List[str]:
    res = []
    for _, df_i in tqdm(df.iterrows(), total=len(df)):
        res_llm = llm_summarization(
            input_text=df_i.inputs
        )
        res.append(res_llm)
    return res

In [10]:
res_model = run_llm_summarization(
    df=df_eval.head(N_MAX_SAMPLES_EVAL),
)
print(f'🤖 {LLM_MODEL_NAME}:\n\n{res_model[:3]} ...\n')

100%|██████████| 109/109 [02:21<00:00,  1.30s/it]


🤖 Falconsai/text_summarization:

['@hulu_support My watchlist is not updating with new episodes (past couple days). Any idea why? @658975 We 
definitely understand, Norlene. For now, we recommend checking the show page for these shows as the new eps will be
there!', '@AirbnbHelp hi, my Acc was linked to an old number. Now I’m asked to verify my Acc, where a code / call 
wil be sent to my old number? Please delete this post as it does have personal info in it.', '@368952 Do you see 
app updates in App Store &gt; Updates? @AppleSupport I am using 11.0.3 and there are no updates for words pro that 
I can find @368952 Thanks forconfirming this.'] ...

In [11]:
df_eval[PRED_COL_NAME] = res_model

In [12]:
model_name_output = LLM_MODEL_NAME.split('/')[0].lower()
output_path = f'../data/processed/df_eval_{model_name_output}.csv'

df_eval.to_csv(output_path, index=False)

print(f'CSV exportado: {output_path}')

CSV exportado: ../data/processed/df_eval_falconsai.csv

## Calcula métricas

In [14]:
print(df_eval.shape)
df_eval.head(1)

(109, 7)

,conversation_id,tweet_ids,created_at_list,inputs,human_summary,elapsed_time,summary_pred
0,bbde6d8ec7c39c4551da1ff6024f997b,"[2263653, 2263654, 2263656, 2263658]",{2263653: Timestamp('2017-11-10 05:21:36+0000'...,@hulu_support My watchlist is not updating wit...,Customer is complaining that the watchlist is ...,6 days 17:39:16,@hulu_support My watchlist is not updating wit...


### Métrica [ROUGE (Recall-Oriented Understudy for Gisting Evaluation)](https://en.wikipedia.org/wiki/ROUGE_(metric))

In [15]:
def calc_rouge_metric(
    df: pd.DataFrame,
    target_name: str = 'target',
    pred_name: str = 'pred'
) -> pd.DataFrame:
    rouge_metric = evaluate.load('rouge')

    df_hf = Dataset.from_pandas(df)

    score = rouge_metric.compute(
        predictions=df_hf[target_name],
        references=df_hf[pred_name]
    )

    df_metrics = pd.DataFrame({LLM_MODEL_NAME: score})
    return df_metrics

In [16]:
df_metrics = calc_rouge_metric(df=df_eval, target_name='human_summary', pred_name=PRED_COL_NAME)
df_metrics = df_metrics.T.reset_index().rename(columns={'index': 'model'})
df_metrics

,model,rouge1,rouge2,rougeL,rougeLsum
0,Falconsai/text_summarization,0.263592,0.108539,0.214594,0.213975


### Exporta métricas em CSV

In [18]:
model_name_output = LLM_MODEL_NAME.split('/')[0].lower().replace('-', '_')
output_path = f'../data/processed/df_metrics_{model_name_output}.csv'

df_metrics.to_csv(output_path, index=False)

print(f'CSV exportado: {output_path}')

CSV exportado: ../data/processed/df_metrics_falconsai.csv